# Graduate admission with an ANN in Keras
Regression: predict a student's chance of admission (0 to 1) from 7 inputs. A linear output node and mean squared error loss.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's log messages
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # take GPU memory only as needed
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# same random numbers on every run
keras.utils.set_random_seed(1)
tf.config.experimental.enable_op_determinism()

## The data

In [ ]:
df = pd.read_csv("data/Admission_Predict_Ver1.1.csv")
# two column names end in a space ("LOR ", "Chance of Admit "): strip them
df.columns = df.columns.str.strip()
print(df.shape)
df.head()

In [ ]:
df.info()
print("duplicates:", df.duplicated().sum())
df.describe().round(2)

## Preparing the inputs

In [ ]:
# the serial number is only a row label
df = df.drop(columns="Serial No.")
X = df.iloc[:, 0:-1]     # 7 input columns
y = df.iloc[:, -1]       # Chance of Admit
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print(X_train.shape, X_test.shape)

In [ ]:
# min-max scaling: every input into 0..1, using the training set's min and max
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled[:2].round(2)

## First network: 7 inputs, 7 ReLU nodes, 1 linear output, 10 epochs

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(7,)),
    layers.Dense(7, activation="relu"),
    layers.Dense(1, activation="linear"),   # any number: regression
])
model.summary()

In [ ]:
model.compile(loss="mean_squared_error", optimizer="adam")
history1 = model.fit(X_train_scaled, y_train, epochs=10, validation_split=0.2, verbose=2)

In [ ]:
y_pred1 = model.predict(X_test_scaled, verbose=0)
print(y_pred1[:5].ravel().round(3), y_test.values[:5])
r2_1 = r2_score(y_test, y_pred1)
print("R2:", r2_1)

## Second network: one more hidden layer, 100 epochs

In [ ]:
model2 = keras.Sequential([
    keras.Input(shape=(7,)),
    layers.Dense(7, activation="relu"),
    layers.Dense(7, activation="relu"),
    layers.Dense(1, activation="linear"),
])
model2.summary()

In [ ]:
model2.compile(loss="mean_squared_error", optimizer="adam")
history = model2.fit(X_train_scaled, y_train, epochs=100, validation_split=0.2, verbose=2)

In [ ]:
y_pred2 = model2.predict(X_test_scaled, verbose=0).ravel()
r2_2 = r2_score(y_test, y_pred2)
print("R2:", r2_2)

In [ ]:
# for comparison: plain linear regression on the same split
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(X_train_scaled, y_train)
r2_lr = r2_score(y_test, lr.predict(X_test_scaled))
print("linear regression R2:", r2_lr)

## Training curves

In [ ]:
h = history.history
epochs = list(range(1, len(h["loss"]) + 1))
fig = go.Figure()
fig.add_scatter(x=epochs, y=h["loss"], name="training loss")
fig.add_scatter(x=epochs, y=h["val_loss"], name="validation loss")
fig.update_layout(template="simple_white", xaxis_title="epoch", yaxis_title="mean squared error")
fig.show()
print({k: round(v[-1], 4) for k, v in h.items()})
print("first network, last epoch:", {k: round(v[-1], 4) for k, v in history1.history.items()})

In [ ]:
# save the numbers the Note's figures are drawn from (images/figs.py)
results = {"history1": history1.history, "history": h, "r2_1": r2_1, "r2_2": r2_2, "r2_lr": r2_lr,
           "y_test": y_test.tolist(), "y_pred1": y_pred1.ravel().tolist(), "y_pred2": y_pred2.tolist()}
Path("images/results.json").write_text(json.dumps(results))